# 01 - Data Cleaning

## Dataset paths

In [ ]:
import os
from PIL import Image
import shutil
import hashlib
from collections import defaultdict

BASE_PATH = os.path.join("..", "..", "datasets", "b_classification_flowers")
DATASET_PATH = os.path.join(BASE_PATH, "b_classification_flowers_original")
CLEAN_PATH = os.path.join(BASE_PATH, "b_classification_flowers_clean")

print(os.listdir(DATASET_PATH))

## Know content of each class (before cleaning)

In [ ]:
for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if os.path.isdir(class_path):
        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        print(f"{class_name}: {len(images)} images")

## Find unreadable images

In [ ]:
bad_images = []

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        try:
            with Image.open(img_path) as img:
                img.convert("RGB").load()
        except (OSError, ValueError):
            bad_images.append(img_path)
            print("Unreadable:", img_path)

print("Total unreadable images:", len(bad_images))

## Find duplicated images (all classes)

In [ ]:
hashes = defaultdict(list)

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in sorted(os.listdir(class_path)):
        img_path = os.path.join(class_path, img_name)

        if not os.path.isfile(img_path):
            continue

        with open(img_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        hashes[file_hash].append(img_path)

duplicate_files = []

for group in hashes.values():
    if len(group) > 1:
        print("Duplicates:", group)
        # keep the first copy, the rest are duplicates
        duplicate_files.extend(group[1:])

print("Total duplicate copies:", len(duplicate_files))

## Copy the good images to the clean folder

In [ ]:
removed = bad_images + duplicate_files
os.makedirs(CLEAN_PATH, exist_ok=True)

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    clean_class_path = os.path.join(CLEAN_PATH, class_name)
    os.makedirs(clean_class_path, exist_ok=True)

    for img_name in os.listdir(class_path):
        src = os.path.join(class_path, img_name)

        if src in removed:
            continue

        dst = os.path.join(clean_class_path, img_name)
        shutil.copy2(src, dst)

print("Clean folder created at:", CLEAN_PATH)

## Save the list of removed files

In [ ]:
os.makedirs(CLEAN_PATH, exist_ok=True)
report_path = os.path.join(CLEAN_PATH, "removed_files.txt")

with open(report_path, "w") as f:
    f.write("Files removed during data cleaning\n")
    f.write("=================================\n\n")

    for path in bad_images:
        f.write(f"UNREADABLE: {path}\n")

    for path in duplicate_files:
        f.write(f"DUPLICATE: {path}\n")

print("Report saved to:", report_path)

## Count images per class (after cleaning)

In [ ]:
for class_name in sorted(name for name in os.listdir(CLEAN_PATH) if os.path.isdir(os.path.join(CLEAN_PATH, name))):
    class_path = os.path.join(CLEAN_PATH, class_name)

    if os.path.isdir(class_path):
        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        print(f"{class_name}: {len(images)} images")